# RKD All-Datasets Runner

This notebook reproduces the behavior of the three example scripts in one place:
- `examples/cub200.sh`
- `examples/cars196.sh`
- `examples/stanford.sh`

For each dataset, it runs:
1. Teacher training (`run.py`)
2. Student distillation with distance+angle (`run_distill.py`)
3. Student distillation with quadruplet-only (`run_distill.py`, `quad_ratio=1`)
4. Self-distillation with distance+angle
5. Self-distillation with quadruplet-only

In [ ]:
from pathlib import Path
import os

# Resolve RKD working directory robustly (local or Colab).
cwd = Path.cwd()
if (cwd / "run.py").exists() and (cwd / "run_distill.py").exists():
    rkd_dir = cwd
elif (cwd / "RKD" / "run.py").exists() and (cwd / "RKD" / "run_distill.py").exists():
    rkd_dir = cwd / "RKD"
else:
    raise FileNotFoundError(
        "Could not find RKD/run.py and RKD/run_distill.py from current directory."
    )

os.chdir(rkd_dir)
print(f"Using RKD directory: {rkd_dir}")

# Prefer repo-level data directory if present (../data), otherwise fallback to RKD/data.
if (rkd_dir.parent / "data").exists():
    DATA_DIR = str(rkd_dir.parent / "data")
else:
    DATA_DIR = str(rkd_dir / "data")

print(f"Using data directory: {DATA_DIR}")

In [ ]:
import run as teacher_runner
import run_distill as distill_runner

# W&B setup (set WANDB_MODE='disabled' if you do not want logging).
WANDB_PROJECT = "rkd-metric-learning"
WANDB_ENTITY = ""
WANDB_MODE = "online"

DATASET_CONFIGS = {
    "cub200": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "embedding_size": 512,
            "save_dir": "cub200_resnet50_512",
        },
        "distill": {
            "epochs": 80,
            "lr_decay_epochs": [40, 60],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 100,
            "student_small_embedding": 128,
        },
    },
    "cars196": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "embedding_size": 512,
            "save_dir": "cars196_resnet50_512",
        },
        "distill": {
            "epochs": 80,
            "lr_decay_epochs": [40, 60],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 100,
            "student_small_embedding": 128,
        },
    },
    "stanford": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "iter_per_epoch": 1000,
            "embedding_size": 512,
            "save_dir": "stanford_resnet50_512",
        },
        "distill": {
            "epochs": 120,
            "lr_decay_epochs": [40, 80],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 500,
            "student_small_embedding": 64,
        },
    },
}

In [ ]:
def _inject_wandb(params, run_name):
    params["wandb_project"] = WANDB_PROJECT
    params["wandb_run_name"] = run_name
    params["wandb_mode"] = WANDB_MODE
    if WANDB_ENTITY.strip():
        params["wandb_entity"] = WANDB_ENTITY


def run_dataset(dataset_name):
    cfg = DATASET_CONFIGS[dataset_name]
    t_cfg = cfg["teacher"]
    d_cfg = cfg["distill"]

    teacher_dir = t_cfg["save_dir"]
    teacher_ckpt = f"{teacher_dir}/best.pth"

    print(f"===== [{dataset_name}] Teacher =====")
    teacher_params = {
        "mode": "train",
        "dataset": dataset_name,
        "base": "resnet50",
        "sample": "distance",
        "margin": 0.2,
        "embedding_size": t_cfg["embedding_size"],
        "epochs": t_cfg["epochs"],
        "lr_decay_epochs": t_cfg["lr_decay_epochs"],
        "lr_decay_gamma": t_cfg["lr_decay_gamma"],
        "batch": t_cfg["batch"],
        "data": DATA_DIR,
        "save_dir": teacher_dir,
    }
    if "iter_per_epoch" in t_cfg:
        teacher_params["iter_per_epoch"] = t_cfg["iter_per_epoch"]
    _inject_wandb(teacher_params, f"{dataset_name}-teacher-resnet50-512")
    teacher_runner.run_with_params(teacher_params)

    print(f"===== [{dataset_name}] Student (resnet18, dist+angle) =====")
    student_da_params = {
        "dataset": dataset_name,
        "base": "resnet18",
        "embedding_size": d_cfg["student_small_embedding"],
        "l2normalize": "false",
        "dist_ratio": 1,
        "angle_ratio": 2,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": DATA_DIR,
        "save_dir": f"{dataset_name}_student_resnet18_{d_cfg['student_small_embedding']}",
    }
    _inject_wandb(student_da_params, f"{dataset_name}-student-resnet18-dist-angle")
    distill_runner.run_with_params(student_da_params)

    print(f"===== [{dataset_name}] Student (resnet18, quadruplet-only) =====")
    student_quad_params = {
        "dataset": dataset_name,
        "base": "resnet18",
        "embedding_size": d_cfg["student_small_embedding"],
        "l2normalize": "false",
        "quad_ratio": 1,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": DATA_DIR,
        "save_dir": f"{dataset_name}_student_resnet18_{d_cfg['student_small_embedding']}_quad",
    }
    _inject_wandb(student_quad_params, f"{dataset_name}-student-resnet18-quad")
    distill_runner.run_with_params(student_quad_params)

    print(f"===== [{dataset_name}] Self-distill (resnet50, dist+angle) =====")
    self_da_params = {
        "dataset": dataset_name,
        "base": "resnet50",
        "embedding_size": 512,
        "l2normalize": "false",
        "dist_ratio": 1,
        "angle_ratio": 2,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": DATA_DIR,
        "save_dir": f"{dataset_name}_student_resnet50_512",
    }
    _inject_wandb(self_da_params, f"{dataset_name}-student-resnet50-dist-angle")
    distill_runner.run_with_params(self_da_params)

    print(f"===== [{dataset_name}] Self-distill (resnet50, quadruplet-only) =====")
    self_quad_params = {
        "dataset": dataset_name,
        "base": "resnet50",
        "embedding_size": 512,
        "l2normalize": "false",
        "quad_ratio": 1,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": DATA_DIR,
        "save_dir": f"{dataset_name}_student_resnet50_512_quad",
    }
    _inject_wandb(self_quad_params, f"{dataset_name}-student-resnet50-quad")
    distill_runner.run_with_params(self_quad_params)

In [ ]:
# Runs all experiments in the same order as the three example scripts.
for dataset_name in ["cub200", "cars196", "stanford"]:
    run_dataset(dataset_name)